* XGBoost 버전 확인

In [ ]:
# xgboost 라이브러리를 불러온다.
# 현재 설치된 XGBoost의 버전을 확인하기 위해 사용한다.
import xgboost

# 설치된 XGBoost 라이브러리의 버전을 출력한다.
print(xgboost.__version__)

### 파이썬 래퍼 XGBoost 적용 – 위스콘신 유방암 예측

In [ ]:
# XGBoost의 파이썬 래퍼를 사용하기 위해 xgboost를 xgb라는 이름으로 불러온다.
import xgboost as xgb

# XGBoost 모델에서 각 피처의 중요도를 시각화하기 위해 plot_importance를 불러온다.
from xgboost import plot_importance

# 데이터프레임 형태로 데이터를 다루기 위해 pandas를 불러온다.
import pandas as pd

# 수치 계산 및 배열 처리를 위해 numpy를 불러온다.
import numpy as np

# 사이킷런에서 제공하는 위스콘신 유방암 데이터셋을 불러오기 위해 사용한다.
from sklearn.datasets import load_breast_cancer

# 학습 데이터와 테스트 데이터를 나누기 위해 train_test_split을 불러온다.
from sklearn.model_selection import train_test_split

# 경고 메시지를 숨기기 위해 warnings 모듈을 불러온다.
import warnings
warnings.filterwarnings('ignore')

# 위스콘신 유방암 데이터셋을 불러온다.
dataset = load_breast_cancer()

# 입력 변수, 즉 feature 데이터를 X_features에 저장한다.
X_features= dataset.data

# 정답 레이블 데이터를 y_label에 저장한다.
y_label = dataset.target

# feature 데이터를 DataFrame 형태로 변환한다.
# columns에는 데이터셋에 포함된 feature 이름을 사용한다.
cancer_df = pd.DataFrame(data=X_features, columns=dataset.feature_names)

# target 컬럼을 추가하여 정답 레이블을 DataFrame에 포함시킨다.
cancer_df['target']= y_label

# 데이터프레임의 앞 3개 행을 확인한다.
cancer_df.head(3)

In [ ]:
# target 값이 의미하는 클래스 이름을 출력한다.
# 위스콘신 유방암 데이터셋에서는 malignant와 benign을 의미한다.
print(dataset.target_names)

# target 컬럼의 값 분포를 확인한다.
# 각 클래스가 몇 개씩 존재하는지 확인하여 데이터 불균형 여부를 볼 수 있다.
print(cancer_df['target'].value_counts())

In [ ]:
# cancer_df에서 feature용 DataFrame과 Label용 Series 객체 추출
# 맨 마지막 칼럼이 Label임. Feature용 DataFrame은 cancer_df의 첫번째 칼럼에서 맨 마지막 두번째 칼럼까지를 :-1 슬라이싱으로 추출.
X_features = cancer_df.iloc[:, :-1]
y_label = cancer_df.iloc[:, -1]

# 전체 데이터 중 80%는 학습용 데이터, 20%는 테스트용 데이터 추출
# random_state를 설정하여 실행할 때마다 동일한 데이터 분할이 이루어지도록 한다.
X_train, X_test, y_train, y_test=train_test_split(X_features, y_label,
                                         test_size=0.2, random_state=156 )

# 위에서 만든 X_train, y_train을 다시 쪼개서 90%는 학습과 10%는 검증용 데이터로 분리
# 검증 데이터는 XGBoost 학습 과정에서 성능 평가 및 조기 중단에 사용된다.
X_tr, X_val, y_tr, y_val= train_test_split(X_train, y_train, test_size=0.1, random_state=156 )

# 학습 데이터와 테스트 데이터의 크기를 출력한다.
print(X_train.shape , X_test.shape)

# 최종 학습 데이터와 검증 데이터의 크기를 출력한다.
print(X_tr.shape, X_val.shape)

In [ ]:
# 만약 구버전 XGBoost에서 DataFrame으로 DMatrix 생성이 안될 경우 X_train.values로 넘파이 변환.
# 학습, 검증, 테스트용 DMatrix를 생성.
# DMatrix는 XGBoost에서 학습 속도와 효율을 높이기 위해 사용하는 전용 데이터 구조이다.
dtr = xgb.DMatrix(data=X_tr, label=y_tr)
dval = xgb.DMatrix(data=X_val, label=y_val)
dtest = xgb.DMatrix(data=X_test , label=y_test)

In [ ]:
# XGBoost 모델 학습에 사용할 하이퍼파라미터를 설정한다.
params = {
    # max_depth는 각 결정 트리의 최대 깊이를 의미한다.
    # 값이 클수록 복잡한 모델이 되지만 과적합 위험이 증가할 수 있다.
    'max_depth':3,

    # eta는 learning_rate와 같은 의미로, 각 트리가 모델에 반영되는 정도를 조절한다.
    # 값이 작을수록 학습은 느리지만 더 안정적으로 학습할 수 있다.
    'eta': 0.05,

    # objective는 학습 목적 함수를 의미한다.
    # binary:logistic은 이진 분류 문제에서 로지스틱 회귀 방식의 확률값을 출력한다.
    'objective':'binary:logistic',

    # eval_metric은 모델 평가 지표를 의미한다.
    # logloss는 분류 모델의 예측 확률이 실제 정답과 얼마나 차이 나는지 평가한다.
    'eval_metric':'logloss'
}

# 부스팅 반복 횟수를 설정한다.
# 즉, 순차적으로 생성할 트리의 최대 개수를 의미한다.
num_rounds = 400

In [ ]:
# 학습 데이터 셋은 'train' 또는 평가 데이터 셋은 'eval' 로 명기합니다.
# eval_list는 학습 과정에서 성능을 평가할 데이터셋 목록이다.
# dtr은 학습 데이터, dval은 검증 데이터이다.
eval_list = [(dtr,'train'),(dval,'eval')] # 또는 eval_list = [(dval,'eval')] 만 명기해도 무방.

# 하이퍼 파라미터와 early stopping 파라미터를 train( ) 함수의 파라미터로 전달
# params에는 앞에서 설정한 XGBoost 하이퍼파라미터가 들어간다.
# dtrain에는 학습 데이터인 dtr을 입력한다.
# num_boost_round는 최대 부스팅 반복 횟수이다.
# early_stopping_rounds=50은 검증 성능이 50번 반복 동안 개선되지 않으면 학습을 조기 중단한다는 의미이다.
# evals에는 학습 중 평가할 데이터셋 목록을 입력한다.
xgb_model = xgb.train(params = params , dtrain=dtr , num_boost_round=num_rounds , \
                      early_stopping_rounds=50, evals=eval_list )

In [ ]:
# 학습된 XGBoost 모델을 이용해 테스트 데이터에 대한 예측 확률을 계산한다.
# binary:logistic을 사용했기 때문에 predict 결과는 0 또는 1이 아니라 클래스 1에 속할 확률로 출력된다.
pred_probs = xgb_model.predict(dtest)

# 예측 확률값 중 앞의 10개만 출력한다.
print('predict( ) 수행 결과값을 10개만 표시, 예측 확률 값으로 표시됨')
print(np.round(pred_probs[:10],3))

# 예측 확률이 0.5 보다 크면 1 , 그렇지 않으면 0 으로 예측값 결정하여 List 객체인 preds에 저장
# 즉, 확률값을 실제 분류 클래스 값으로 변환하는 과정이다.
preds = [ 1 if x > 0.5 else 0 for x in pred_probs ]

# 변환된 예측 클래스값 중 앞의 10개만 출력한다.
print('예측값 10개만 표시:',preds[:10])

In [ ]:
# 오차 행렬을 계산하기 위해 confusion_matrix를 불러온다.
# 정확도를 계산하기 위해 accuracy_score를 불러온다.
from sklearn.metrics import confusion_matrix, accuracy_score

# 정밀도와 재현율을 계산하기 위해 precision_score, recall_score를 불러온다.
from sklearn.metrics import precision_score, recall_score

# F1 score와 ROC-AUC를 계산하기 위해 f1_score, roc_auc_score를 불러온다.
from sklearn.metrics import f1_score, roc_auc_score

# 분류 모델의 여러 평가 지표를 한 번에 출력하는 함수를 정의한다.
def get_clf_eval(y_test, pred=None, pred_proba=None):
    # 오차 행렬을 계산한다.
    # 실제값과 예측값을 비교하여 TN, FP, FN, TP 개수를 확인할 수 있다.
    confusion = confusion_matrix( y_test, pred)

    # 정확도는 전체 데이터 중 올바르게 예측한 비율이다.
    accuracy = accuracy_score(y_test , pred)

    # 정밀도는 양성으로 예측한 것 중 실제 양성의 비율이다.
    precision = precision_score(y_test , pred)

    # 재현율은 실제 양성 중 모델이 양성으로 맞게 예측한 비율이다.
    recall = recall_score(y_test , pred)

    # F1 score는 정밀도와 재현율의 조화 평균이다.
    f1 = f1_score(y_test,pred)

    # ROC-AUC 추가
    # ROC-AUC는 분류 모델이 양성과 음성을 얼마나 잘 구분하는지 나타내는 지표이다.
    roc_auc = roc_auc_score(y_test, pred_proba)

    # 오차 행렬을 출력한다.
    print('오차 행렬')
    print(confusion)

    # ROC-AUC print 추가
    # 정확도, 정밀도, 재현율, F1, AUC를 한 번에 출력한다.
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f},\
    F1: {3:.4f}, AUC:{4:.4f}'.format(accuracy, precision, recall, f1, roc_auc))

In [ ]:
# 테스트 데이터의 실제값, 예측 클래스값, 예측 확률값을 이용하여
# XGBoost 모델의 분류 성능을 평가한다.
get_clf_eval(y_test , preds, pred_probs)

In [ ]:
# 그래프 시각화를 위해 matplotlib.pyplot을 불러온다.
import matplotlib.pyplot as plt

# 주피터 노트북 내부에 그래프가 바로 출력되도록 설정한다.
%matplotlib inline

# 피처 중요도 그래프의 크기를 설정한다.
fig, ax = plt.subplots(figsize=(10, 12))

# 학습된 XGBoost 모델의 피처 중요도를 시각화한다.
# 어떤 변수가 예측에 상대적으로 크게 기여했는지 확인할 수 있다.
plot_importance(xgb_model, ax=ax)

# 생성된 피처 중요도 그래프를 tif 파일로 저장한다.
plt.savefig('p239_xgb_feature_importance.tif', format='tif', dpi=300, bbox_inches='tight')

### 사이킷런 래퍼 XGBoost의 개요 및 적용

In [ ]:
# 사이킷런 래퍼 XGBoost 클래스인 XGBClassifier 임포트
from xgboost import XGBClassifier

# XGBClassifier 객체를 생성한다.
# n_estimators는 생성할 트리의 개수이다.
# learning_rate는 각 트리가 모델에 반영되는 비율이다.
# max_depth는 각 트리의 최대 깊이이다.
xgb_wrapper = XGBClassifier(n_estimators=400, learning_rate=0.1, max_depth=3)

# 학습 데이터를 이용하여 사이킷런 래퍼 방식의 XGBoost 모델을 학습시킨다.
xgb_wrapper.fit(X_train, y_train)

# 학습된 모델을 이용하여 테스트 데이터의 클래스를 예측한다.
w_preds = xgb_wrapper.predict(X_test)

# 테스트 데이터가 클래스 1에 속할 예측 확률값을 추출한다.
# predict_proba 결과에서 두 번째 컬럼이 클래스 1의 확률이다.
w_pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]

In [ ]:
# 사이킷런 래퍼 XGBoost 모델의 성능을 평가한다.
# 실제값, 예측값, 예측 확률값을 이용하여 오차 행렬, 정확도, 정밀도, 재현율, F1, AUC를 출력한다.
get_clf_eval(y_test , w_preds, w_pred_proba)

In [ ]:
# 사이킷런 래퍼 XGBoost 클래스인 XGBClassifier를 다시 불러온다.
from xgboost import XGBClassifier

# XGBClassifier 객체를 생성한다.
# n_estimators=400은 최대 400개의 트리를 생성한다는 의미이다.
# learning_rate=0.1은 각 트리의 반영 비율을 의미한다.
# max_depth=3은 각 트리의 최대 깊이를 3으로 제한한다는 의미이다.
xgb_wrapper = XGBClassifier(n_estimators=400, learning_rate=0.1, max_depth=3)

# 검증 데이터셋을 설정한다.
# 여기서는 테스트 데이터를 평가용 데이터셋으로 지정하였다.
evals = [(X_test, y_test)]

# 사이킷런 래퍼 방식의 XGBoost 모델을 학습한다.
# early_stopping_rounds=100은 평가 지표가 100번 반복 동안 개선되지 않으면 학습을 조기 중단한다는 의미이다.
# eval_metric="logloss"는 평가 지표로 로그 손실을 사용한다는 의미이다.
# eval_set=evals는 학습 과정에서 평가할 데이터셋을 지정하는 부분이다.
# verbose=True는 학습 과정의 평가 결과를 출력하도록 한다.
xgb_wrapper.fit(X_train, y_train, early_stopping_rounds=100, eval_metric="logloss",
                eval_set=evals, verbose=True)

# 조기 중단 조건을 적용한 모델로 테스트 데이터의 클래스를 예측한다.
ws100_preds = xgb_wrapper.predict(X_test)

# 테스트 데이터가 클래스 1에 속할 예측 확률값을 추출한다.
ws100_pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]

In [ ]:
# early_stopping_rounds=100을 적용한 XGBoost 모델의 성능을 평가한다.
get_clf_eval(y_test , ws100_preds, ws100_pred_proba)

In [ ]:
# early_stopping_rounds를 10으로 설정하고 재 학습.
# 조기 중단 기준을 더 짧게 설정하여 모델을 다시 학습한다.
# 성능이 10번 반복 동안 개선되지 않으면 학습을 멈춘다.
xgb_wrapper.fit(X_train, y_train, early_stopping_rounds=10,
                eval_metric="logloss", eval_set=evals,verbose=True)

# early_stopping_rounds=10을 적용한 모델로 테스트 데이터의 클래스를 예측한다.
ws10_preds = xgb_wrapper.predict(X_test)

# 테스트 데이터가 클래스 1에 속할 예측 확률값을 추출한다.
ws10_pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]

# early_stopping_rounds=10을 적용한 모델의 성능을 평가한다.
get_clf_eval(y_test , ws10_preds, ws10_pred_proba)

In [ ]:
# XGBoost의 피처 중요도를 시각화하기 위해 plot_importance를 불러온다.
from xgboost import plot_importance

# 그래프 시각화를 위해 matplotlib.pyplot을 불러온다.
import matplotlib.pyplot as plt

# 주피터 노트북 내부에 그래프가 바로 출력되도록 설정한다.
%matplotlib inline

# 피처 중요도 그래프의 크기를 설정한다.
fig, ax = plt.subplots(figsize=(10, 12))

# 사이킷런 래퍼 클래스를 입력해도 무방.
# 학습된 XGBClassifier 모델의 피처 중요도를 시각화한다.
plot_importance(xgb_wrapper, ax=ax)